# Build Current Hourly Station Flow from GBFS Snapshots

This notebook builds the **current hourly station-level flow table** used as the main operational input for downstream forecasting jobs.

Using raw GBFS station snapshots, it estimates:

- hourly departures
- hourly arrivals
- end-of-hour bike and dock availability

The notebook then enriches the hourly station records with:

- station metadata
- weather conditions
- event-related features
- quality flags

Its main objective is to generate a consistent and up-to-date table:

- `workspace.default.station_hour_flow_current`

This table acts as the **live operational feature layer** for both:
- the 1-hour ahead prediction pipeline
- the multi-hour / 1-week forecasting pipeline

## Process Overview

This notebook performs the following steps:

### 1. Read GBFS station snapshots
The job loads raw station status records from the bronze GBFS status table and restricts the dataset to a recent time window for efficiency.

### 2. Convert timestamps to local hourly structure
UTC ingestion timestamps are transformed into local Toronto time so snapshots can be grouped consistently by:
- date
- year
- month
- day
- hour

### 3. Estimate 5-minute station flow
Bike availability changes between consecutive GBFS snapshots are used to estimate:
- departures
- arrivals

These are treated as short-interval flow approximations.

### 4. Aggregate 5-minute estimates into hourly station flow
The notebook aggregates the short-interval estimates into hourly totals and identifies the final station state observed within each hour.

### 5. Join static station metadata
Station information such as:
- name
- coordinates
- capacity

is merged from the latest station information view.

### 6. Filter to downtown Toronto stations
Only the downtown operating area used in model training is retained to ensure consistency with the historical Gold V2 training data.

### 7. Join hourly weather
Weather conditions for the corresponding station-hour are added from the latest hourly weather view.

### 8. Build calendar attributes
Date-based variables are created, including:
- date
- day of week
- weekend indicator

### 9. Join hourly event signals
Public event records are spatially matched to each station-hour using haversine distance and event spillover logic.

The notebook derives:
- day-level event features
- nearby hourly event features
- attendance-based impact features

### 10. Create quality flags
Operational quality checks are applied to identify:
- whether the hour is sufficiently complete
- whether enough valid snapshot intervals exist

### 11. Save the operational hourly table
The final enriched output is written to:
- `workspace.default.station_hour_flow_current`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# Build hourly estimated departures/arrivals from GBFS snapshots
# Uses:
#   - current GBFS bronze
#   - latest station info
#   - latest weather
#   - public_events_hourly_detail
# Output:
#   - workspace.default.station_flow_5min_estimated
#   - workspace.default.station_hour_flow_current
# ============================================================

STATUS_TBL  = "workspace.default.bronze_gbfs_station_status"
STATION_VW  = "workspace.default.vw_gbfs_station_information_latest"
WEATHER_VW  = "workspace.default.vw_weather_hourly_minimal_latest"

EVENTS_DETAIL_DIR = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver_agg/"
    "public_events_hourly_detail"
)

FLOW_5M_TBL = "workspace.default.station_flow_5min_estimated"
FLOW_HR_TBL = "workspace.default.station_hour_flow_current"

# ------------------------------------------------------------
# Downtown bounding box (same as historical Gold_v2 training)
# ------------------------------------------------------------
DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX = 43.63, 43.67
DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX = -79.41, -79.37

# ------------------------------------------------------------
# Event influence config
# ------------------------------------------------------------
EVENT_RADIUS_KM = 3.0
EVENT_DECAY_KM = 1.5
EVENT_EPS = 0.10

VERY_HIGH_RADIUS_KM = 0.3
HIGH_RADIUS_KM = 0.6
MEDIUM_RADIUS_KM = 1.5
LOW_RADIUS_KM = 3.0

VERY_HIGH_MULT = 1.8
HIGH_MULT = 1.4
MEDIUM_MULT = 1.0
LOW_MULT = 0.6

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def haversine_expr(lat1_col, lon1_col, lat2_col, lon2_col):
    """
    Spark expression for haversine distance in km.
    """
    lat1 = F.radians(F.col(lat1_col))
    lon1 = F.radians(F.col(lon1_col))
    lat2 = F.radians(F.col(lat2_col))
    lon2 = F.radians(F.col(lon2_col))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        F.pow(F.sin(dlat / 2.0), 2) +
        F.cos(lat1) * F.cos(lat2) * F.pow(F.sin(dlon / 2.0), 2)
    )
    c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
    return 6371.0 * c

def spillover_multiplier_expr(dist_col):
    return (
        F.when(F.col(dist_col) <= VERY_HIGH_RADIUS_KM, F.lit(VERY_HIGH_MULT))
         .when(F.col(dist_col) <= HIGH_RADIUS_KM, F.lit(HIGH_MULT))
         .when(F.col(dist_col) <= MEDIUM_RADIUS_KM, F.lit(MEDIUM_MULT))
         .when(F.col(dist_col) <= LOW_RADIUS_KM, F.lit(LOW_MULT))
         .otherwise(F.lit(0.0))
    )

def ensure_event_defaults(df):
    defaults = {
        "event_day_flag": 0,
        "events_day_count": 0,
        "event_day_attendance_sum": 0.0,
        "event_active_nearby_flag": 0,
        "events_nearby_count": 0,
        "nearest_event_km": 999.0,
        "event_weighted_intensity": 0.0,
        "event_attendance_est_sum_nearby": 0.0,
        "event_impact_score": 0.0,
    }
    for c, v in defaults.items():
        if c not in df.columns:
            df = df.withColumn(c, F.lit(v))
        else:
            df = df.withColumn(c, F.coalesce(F.col(c), F.lit(v)))
    return df

# ------------------------------------------------------------
# 1) Read GBFS status
# ------------------------------------------------------------
df_status = spark.sql(f"""
SELECT
    station_id,
    num_bikes_available,
    num_docks_available,
    ingested_at_utc
FROM {STATUS_TBL}
""")

# Limit recent window for performance
df_status = df_status.filter(
    "ingested_at_utc >= current_timestamp() - INTERVAL 30 DAYS"
)

# ------------------------------------------------------------
# 2) Build local timestamp and hour fields
# ------------------------------------------------------------
df_status = (
    df_status
    .withColumn("snapshot_ts_utc", F.col("ingested_at_utc"))
    .withColumn("snapshot_ts_local", F.from_utc_timestamp("ingested_at_utc", "America/Toronto"))
    .withColumn("date_local", F.to_date("snapshot_ts_local"))
    .withColumn("year", F.year("snapshot_ts_local"))
    .withColumn("month", F.month("snapshot_ts_local"))
    .withColumn("day", F.dayofmonth("snapshot_ts_local"))
    .withColumn("hour", F.hour("snapshot_ts_local"))
)

# ------------------------------------------------------------
# 3) Estimate 5-minute departures/arrivals by station
# ------------------------------------------------------------
w_station = Window.partitionBy("station_id").orderBy("snapshot_ts_local")

df_5m = (
    df_status
    .withColumn("prev_bikes_available", F.lag("num_bikes_available", 1).over(w_station))
    .withColumn("prev_snapshot_ts_local", F.lag("snapshot_ts_local", 1).over(w_station))
    .withColumn(
        "minutes_since_prev",
        (F.unix_timestamp("snapshot_ts_local") - F.unix_timestamp("prev_snapshot_ts_local")) / 60.0
    )
)

# Only trust deltas for reasonably close consecutive snapshots
df_5m = df_5m.withColumn(
    "valid_prev_flag",
    F.when(
        (F.col("prev_bikes_available").isNotNull()) &
        (F.col("minutes_since_prev") <= 15),
        1
    ).otherwise(0)
)

df_5m = (
    df_5m
    .withColumn(
        "estimated_departures_5m",
        F.when(
            F.col("valid_prev_flag") == 1,
            F.greatest(F.col("prev_bikes_available") - F.col("num_bikes_available"), F.lit(0))
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "estimated_arrivals_5m",
        F.when(
            F.col("valid_prev_flag") == 1,
            F.greatest(F.col("num_bikes_available") - F.col("prev_bikes_available"), F.lit(0))
        ).otherwise(F.lit(0))
    )
)

# Save 5-min table
(
    df_5m
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FLOW_5M_TBL)
)

print(f"5-minute estimated flow table created: {FLOW_5M_TBL}")

# ------------------------------------------------------------
# 4) Pick end-of-hour station state
# ------------------------------------------------------------
w_hour_last = Window.partitionBy("station_id", "year", "month", "day", "hour").orderBy(F.col("snapshot_ts_local").desc())

df_hour_last = (
    df_5m
    .withColumn("rn", F.row_number().over(w_hour_last))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        "station_id",
        "year",
        "month",
        "day",
        "hour",
        "snapshot_ts_local",
        "num_bikes_available",
        "num_docks_available"
    )
    .withColumnRenamed("snapshot_ts_local", "hour_end_snapshot_ts_local")
    .withColumnRenamed("num_bikes_available", "num_bikes_available_end_hour")
    .withColumnRenamed("num_docks_available", "num_docks_available_end_hour")
)

# ------------------------------------------------------------
# 5) Aggregate 5-minute estimated flow to hourly
# ------------------------------------------------------------
df_hour_flow = (
    df_5m
    .groupBy("station_id", "year", "month", "day", "hour")
    .agg(
        F.sum("estimated_departures_5m").alias("estimated_departures"),
        F.sum("estimated_arrivals_5m").alias("estimated_arrivals"),
        F.count("*").alias("snapshots_in_hour"),
        F.sum("valid_prev_flag").alias("valid_intervals_in_hour"),
        F.min("snapshot_ts_local").alias("first_snapshot_in_hour"),
        F.max("snapshot_ts_local").alias("last_snapshot_in_hour")
    )
)

df_hour_flow = df_hour_flow.join(
    df_hour_last,
    on=["station_id", "year", "month", "day", "hour"],
    how="left"
)

# ------------------------------------------------------------
# 6) Join station metadata
# ------------------------------------------------------------
df_station = spark.sql(f"""
SELECT
    station_id,
    name,
    lat,
    lon,
    capacity
FROM {STATION_VW}
""")

df_hour_flow = df_hour_flow.join(df_station, on="station_id", how="left")

# ------------------------------------------------------------
# 6.1) Filter downtown stations only
# ------------------------------------------------------------
df_hour_flow = df_hour_flow.filter(
    (F.col("lat").between(DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX)) &
    (F.col("lon").between(DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX))
)

# ------------------------------------------------------------
# 7) Join weather
# ------------------------------------------------------------
df_weather = spark.sql(f"""
SELECT
    year,
    month,
    day,
    hour,
    temperature_2m_c AS temperature_2m_celsius,
    apparent_temperature_c AS apparent_temperature_celsius
FROM {WEATHER_VW}
""")

df_hour_flow = df_hour_flow.join(
    df_weather,
    on=["year", "month", "day", "hour"],
    how="left"
)

# ------------------------------------------------------------
# 8) Build date / calendar features
# ------------------------------------------------------------
df_hour_flow = (
    df_hour_flow
    .withColumn("date", F.make_date("year", "month", "day"))
    .withColumn("dow_num", F.dayofweek("date"))
    .withColumn("is_weekend", F.when(F.col("dow_num").isin([1, 7]), 1).otherwise(0))
)

# ------------------------------------------------------------
# 9) Join events from public_events_hourly_detail
# Build station-hour event features with spatial impact
# ------------------------------------------------------------
try:
    events = spark.read.parquet(EVENTS_DETAIL_DIR).select(
        "year", "month", "day", "hour",
        "event_id",
        "start_date",
        "end_date",
        "event_lat",
        "event_lon",
        "attendance_est"
    )

    # -------------------------------
    # A) Day-level features
    # -------------------------------
    events_day = (
        events
        .select("year", "month", "day", "event_id", "attendance_est")
        .dropDuplicates(["year", "month", "day", "event_id"])
        .groupBy("year", "month", "day")
        .agg(
            F.lit(1).alias("event_day_flag"),
            F.countDistinct("event_id").alias("events_day_count"),
            F.sum("attendance_est").alias("event_day_attendance_sum")
        )
    )

    # -------------------------------
    # B) Hour-level station-event join
    # -------------------------------
    station_hour_keys = df_hour_flow.select(
        "station_id", "year", "month", "day", "hour", "lat", "lon"
    )

    station_event = (
        station_hour_keys.alias("s")
        .join(
            events.alias("e"),
            on=["year", "month", "day", "hour"],
            how="left"
        )
        .withColumn(
            "distance_km",
            haversine_expr("lat", "lon", "event_lat", "event_lon")
        )
    )

    station_event = station_event.withColumn(
        "within_event_radius_flag",
        F.when(F.col("distance_km") <= EVENT_RADIUS_KM, 1).otherwise(0)
    )

    # Inverse-distance intensity
    station_event = station_event.withColumn(
        "event_weight_component",
        F.when(
            F.col("distance_km") <= EVENT_RADIUS_KM,
            F.col("attendance_est") / (F.col("distance_km") + F.lit(EVENT_EPS))
        ).otherwise(F.lit(0.0))
    )

    # Spillover score
    station_event = station_event.withColumn(
        "spillover_component",
        F.when(
            F.col("distance_km") <= EVENT_RADIUS_KM,
            F.col("attendance_est") *
            F.exp(-F.col("distance_km") / F.lit(EVENT_DECAY_KM)) *
            spillover_multiplier_expr("distance_km")
        ).otherwise(F.lit(0.0))
    )

    station_event = station_event.withColumn(
        "attendance_nearby_component",
        F.when(F.col("distance_km") <= EVENT_RADIUS_KM, F.col("attendance_est")).otherwise(F.lit(0.0))
    )

    # -------------------------------
    # C) Aggregate station-hour event features
    # -------------------------------
    events_station_hour = (
        station_event
        .groupBy("station_id", "year", "month", "day", "hour")
        .agg(
            F.max(
                F.when(F.col("within_event_radius_flag") == 1, F.lit(1)).otherwise(F.lit(0))
            ).alias("event_active_nearby_flag"),

            F.sum(
                F.when(F.col("within_event_radius_flag") == 1, F.lit(1)).otherwise(F.lit(0))
            ).alias("events_nearby_count"),

            F.min(
                F.when(F.col("event_id").isNotNull(), F.col("distance_km"))
            ).alias("nearest_event_km"),

            F.sum("event_weight_component").alias("event_weighted_intensity"),
            F.sum("attendance_nearby_component").alias("event_attendance_est_sum_nearby"),
            F.sum("spillover_component").alias("event_impact_score")
        )
    )

    events_station_hour = events_station_hour.withColumn(
        "nearest_event_km",
        F.coalesce(F.col("nearest_event_km"), F.lit(999.0))
    )

    # -------------------------------
    # D) Join back into main table
    # -------------------------------
    df_hour_flow = (
        df_hour_flow
        .join(events_day, on=["year", "month", "day"], how="left")
        .join(events_station_hour, on=["station_id", "year", "month", "day", "hour"], how="left")
    )

    print(f"Events joined from parquet: {EVENTS_DETAIL_DIR}")

except Exception as e:
    print(f"Events parquet not available or invalid: {e}")

df_hour_flow = ensure_event_defaults(df_hour_flow)

# ------------------------------------------------------------
# 10) Quality flags
# ------------------------------------------------------------
df_hour_flow = (
    df_hour_flow
    .withColumn(
        "hour_complete_flag",
        F.when(F.col("snapshots_in_hour") >= 10, 1).otherwise(0)
    )
    .withColumn(
        "flow_quality_flag",
        F.when(F.col("valid_intervals_in_hour") >= 8, 1).otherwise(0)
    )
    .withColumn("processed_at_utc", F.current_timestamp())
)

# ------------------------------------------------------------
# 11) Save hourly table
# ------------------------------------------------------------
(
    df_hour_flow
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FLOW_HR_TBL)
)

print(f"Hourly estimated flow table created: {FLOW_HR_TBL}")
print(f"Rows: {df_hour_flow.count()}")

## Outputs

This notebook produces two operational Delta tables:

### 1. 5-minute estimated flow table
- `workspace.default.station_flow_5min_estimated`

This table contains short-interval estimates of:
- departures
- arrivals
- snapshot continuity

It is useful for intermediate diagnostics and hourly aggregation.

---

### 2. Current hourly station flow table
- `workspace.default.station_hour_flow_current`

This is the main operational output of the notebook.

It contains:
- estimated hourly arrivals and departures
- end-of-hour bike and dock availability
- station metadata
- weather variables
- event-related spatial features
- calendar variables
- quality flags

This table is consumed directly by:
- the 1-hour prediction job
- the multi-hour forecasting job

## Key Insights and Summary

### 1. GBFS snapshots are transformed into operational flow signals
Raw availability snapshots do not directly provide departures and arrivals. This notebook estimates these flows by comparing bike availability across consecutive short-interval records.

This creates a usable operational demand proxy from real-time system data.

---

### 2. The notebook produces both station state and hourly movement estimates
The output captures two critical aspects of the system:
- the estimated hourly flow
- the final observed bike/dock balance at the end of the hour

This makes the table suitable for both forecasting and operational monitoring.

---

### 3. The live operational dataset is aligned with training logic
By filtering to the same downtown area and enriching the records with weather, events, and calendar features, the notebook ensures consistency between:
- historical training data
- real-time serving data

This is essential for reliable production inference.

---

### 4. Event spillover logic introduces localized urban context
Instead of only counting events by day, the notebook estimates station-level event influence using:
- distance to event
- nearby attendance
- weighted intensity
- spillover impact

This improves the realism of station-hour context.

---

### 5. Quality flags improve operational trust
The notebook explicitly tracks:
- whether the hour has enough snapshots
- whether enough valid intervals are available

These flags are useful for identifying weak or incomplete operational records before prediction.

---

### 6. Business relevance
This notebook converts raw real-time bike system observations into a structured operational dataset that supports:
- short-term forecasting
- station imbalance monitoring
- bike/dock risk estimation
- operational rebalancing decisions

In practical terms, this notebook is the **live feature-construction engine** that connects real-time system data to downstream prediction jobs.